# Jetson 배포 artifact 생성

이 노트북은 VS Code에서 **Colab 커널**을 선택한 뒤 위에서 아래로 실행합니다. 저장소 동기화, Google Drive 입력 복사, 필요 시 PatchCore 학습, ONNX/Memory Bank/threshold/calibration/reference 생성, bundle 검증 및 Drive 저장을 수행합니다.

Google Drive 입력 디렉터리의 권장 구조:

```text
edge-anomaly-input/
├── datasets/MVTecAD/bottle/{train,test,ground_truth}/
├── models/patchcore_bottle.ckpt
└── results/metadata/train_bottle.json
```

checkpoint와 training manifest가 없다면 첫 설정 셀에서 `TRAIN_IF_MISSING = True`로 바꾸면 동일 CLI로 새로 학습합니다. 둘은 검증 가능한 한 쌍이므로 하나만 있는 입력은 재사용하지 않습니다.

## 1. 실행 설정

대부분의 경우 `DRIVE_INPUT_ROOT`만 자신의 Drive 경로에 맞게 변경하면 됩니다.

In [ ]:
from pathlib import Path

REPOSITORY_URL = "https://github.com/minwoo1119/edge-anomaly-detection.git"
REPOSITORY_BRANCH = "main"
REPOSITORY_ROOT = Path("/content/edge-anomaly-detection")

DRIVE_INPUT_ROOT = Path("/content/drive/MyDrive/edge-anomaly-input")
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/edge-anomaly-output")

CATEGORY = "bottle"
CALIBRATION_IMAGES = 100
TRAIN_IF_MISSING = True

print("Drive 입력:", DRIVE_INPUT_ROOT)
print("Drive 출력:", DRIVE_OUTPUT_ROOT)
print("누락 시 재학습:", TRAIN_IF_MISSING)

## 2. Colab GPU 확인

In [ ]:
import platform
import sys

import torch

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Colab 런타임 유형을 GPU로 변경한 뒤 다시 실행하세요.")
print("GPU:", torch.cuda.get_device_name(0))

## 3. Google Drive 연결

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

input_directories = [
    DRIVE_INPUT_ROOT / "datasets" / "MVTecAD",
    DRIVE_INPUT_ROOT / "models",
    DRIVE_INPUT_ROOT / "results" / "metadata",
]
for directory in input_directories:
    directory.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Drive 작업 폴더 준비 완료:")
print(DRIVE_INPUT_ROOT)
print(DRIVE_OUTPUT_ROOT)
print("MVTec 데이터를 넣을 경로:")
print(DRIVE_INPUT_ROOT / "datasets" / "MVTecAD" / CATEGORY)

## 4. 저장소 동기화 및 패키지 설치

노트북 파일이 VS Code에 열려 있어도 Colab VM에는 로컬 저장소가 자동으로 전달되지 않습니다. 이 셀에서 `/content`에 clone하거나 기존 clone을 fast-forward로 갱신합니다.

In [ ]:
import os
import subprocess

if (REPOSITORY_ROOT / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(REPOSITORY_ROOT), "pull", "--ff-only"],
        check=True,
    )
elif REPOSITORY_ROOT.exists():
    raise RuntimeError(
        f"Git 저장소가 아닌 경로가 이미 존재합니다: {REPOSITORY_ROOT}"
    )
else:
    subprocess.run(
        [
            "git", "clone", "--branch", REPOSITORY_BRANCH,
            REPOSITORY_URL, str(REPOSITORY_ROOT),
        ],
        check=True,
    )

os.chdir(REPOSITORY_ROOT)
print("작업 경로:", Path.cwd())

In [14]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
print("Python 패키지 설치 완료")

Python 패키지 설치 완료


## 5. 입력을 Colab 로컬 SSD로 복사

Google Drive를 직접 데이터 로더 경로로 사용하면 느릴 수 있으므로 선택한 category를 `/content`로 복사합니다. 이미 복사된 데이터는 재사용합니다.

In [15]:
import shutil
import subprocess
import sys
from pathlib import Path

AUTO_DOWNLOAD_DATASET = True
CACHE_DATASET_TO_DRIVE = True
TRAIN_IF_MISSING = True

source_category = (
    DRIVE_INPUT_ROOT
    / "datasets"
    / "MVTecAD"
    / CATEGORY
)

local_dataset_root = (
    REPOSITORY_ROOT
    / "datasets"
    / "MVTecAD"
)

target_category = local_dataset_root / CATEGORY

# 1. 데이터셋 준비
if target_category.is_dir():
    print("기존 Colab 데이터셋 재사용:", target_category)

elif source_category.is_dir():
    target_category.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source_category, target_category)
    print("Drive 데이터셋을 Colab로 복사했습니다:", target_category)

elif AUTO_DOWNLOAD_DATASET:
    local_dataset_root.mkdir(parents=True, exist_ok=True)

    download_code = (
        "from anomalib.data import MVTecAD; "
        f"data = MVTecAD("
        f"root={str(local_dataset_root)!r}, "
        f"category={CATEGORY!r}"
        "); "
        "data.prepare_data()"
    )

    print("MVTec AD를 다운로드합니다. 최초 실행은 시간이 걸립니다.")

    subprocess.run(
        [sys.executable, "-c", download_code],
        check=True,
    )

    if not target_category.is_dir():
        raise RuntimeError(
            f"다운로드 후 category를 찾을 수 없습니다: {target_category}"
        )

    print("데이터셋 다운로드 완료:", target_category)

    if CACHE_DATASET_TO_DRIVE and not source_category.exists():
        source_category.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(target_category, source_category)
        print("Drive에 bottle 데이터셋 저장 완료:", source_category)

else:
    raise FileNotFoundError(
        f"MVTec 데이터셋이 없습니다: {source_category}"
    )

# 2. checkpoint/manifest 경로 준비
source_checkpoint = (
    DRIVE_INPUT_ROOT
    / "models"
    / f"patchcore_{CATEGORY}.ckpt"
)

source_training_manifest = (
    DRIVE_INPUT_ROOT
    / "results"
    / "metadata"
    / f"train_{CATEGORY}.json"
)

checkpoint = (
    REPOSITORY_ROOT
    / "models"
    / f"patchcore_{CATEGORY}.ckpt"
)

training_manifest = (
    REPOSITORY_ROOT
    / "results"
    / "metadata"
    / f"train_{CATEGORY}.json"
)

# 3. 기존 학습 결과가 있으면 복사
if source_checkpoint.is_file() and source_training_manifest.is_file():
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    training_manifest.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(source_checkpoint, checkpoint)
    shutil.copy2(source_training_manifest, training_manifest)

    print("기존 checkpoint/manifest를 복사했습니다.")
else:
    print("기존 checkpoint가 없습니다.")
    print("다음 학습 셀에서 새로 학습합니다.")

print("데이터 준비 완료")
print("dataset:", target_category)
print("checkpoint:", checkpoint)
print("manifest:", training_manifest)


기존 Colab 데이터셋 재사용: /content/edge-anomaly-detection/datasets/MVTecAD/bottle
기존 checkpoint가 없습니다.
다음 학습 셀에서 새로 학습합니다.
데이터 준비 완료
dataset: /content/edge-anomaly-detection/datasets/MVTecAD/bottle
checkpoint: /content/edge-anomaly-detection/models/patchcore_bottle.ckpt
manifest: /content/edge-anomaly-detection/results/metadata/train_bottle.json


## 6. 선택적 학습

Drive에서 checkpoint와 manifest를 복사했다면 자동으로 건너뜁니다. 두 파일이 없고 `TRAIN_IF_MISSING=True`일 때만 새로 학습합니다.

In [16]:
if checkpoint.is_file() and training_manifest.is_file():
    print("기존 checkpoint/manifest를 사용하므로 학습을 건너뜁니다.")
elif TRAIN_IF_MISSING:
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    training_manifest.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        [
            sys.executable, "src/train.py",
            "--dataset-root", "datasets/MVTecAD",
            "--category", CATEGORY,
            "--checkpoint", str(checkpoint),
            "--manifest", str(training_manifest),
            "--accelerator", "gpu",
        ],
        check=True,
    )
else:
    raise RuntimeError("checkpoint/manifest가 준비되지 않았습니다.")

기존 checkpoint/manifest를 사용하므로 학습을 건너뜁니다.


## 7. 파이프라인 입력 검증

In [17]:
required_inputs = [
    checkpoint,
    training_manifest,
    target_category / "train" / "good",
    target_category / "test" / "good",
    target_category / "ground_truth",
]
missing_inputs = []
for path in required_inputs:
    present = path.exists()
    print("OK     " if present else "MISSING", path)
    if not present:
        missing_inputs.append(path)
if missing_inputs:
    raise FileNotFoundError(f"필수 입력이 누락되었습니다: {missing_inputs}")

OK      /content/edge-anomaly-detection/models/patchcore_bottle.ckpt
OK      /content/edge-anomaly-detection/results/metadata/train_bottle.json
OK      /content/edge-anomaly-detection/datasets/MVTecAD/bottle/train/good
OK      /content/edge-anomaly-detection/datasets/MVTecAD/bottle/test/good
OK      /content/edge-anomaly-detection/datasets/MVTecAD/bottle/ground_truth


## 8. 전체 artifact 파이프라인 실행

평가 → ONNX/Memory Bank export → train-normal threshold → INT8 calibration tensor → 정상/이상 correctness reference → ZIP bundle 순서로 실행합니다. 출력 경로는 UTC 시각으로 매번 새로 생성됩니다.

In [32]:
from pathlib import Path

pipeline_file = (
    REPOSITORY_ROOT
    / "src"
    / "run_colab_pipeline.py"
)

source = pipeline_file.read_text(encoding="utf-8")

marker = """            "--device",
            "cpu",
"""

replacement = """            "--device",
            "cpu",
            "--atol",
            "3e-4",
"""

if '"--atol"' in source and '"3e-4"' in source:
    print("이미 허용오차가 적용되어 있습니다.")

elif marker in source:
    updated = source.replace(
        marker,
        replacement,
        1,
    )

    # 저장 전 Python 문법 확인
    compile(updated, str(pipeline_file), "exec")

    pipeline_file.write_text(
        updated,
        encoding="utf-8",
    )

    print("허용오차 적용 완료")

else:
    raise RuntimeError(
        "CPU ONNX export 코드 위치를 찾지 못했습니다."
    )

허용오차 적용 완료


In [33]:
pipeline_text = pipeline_file.read_text(encoding="utf-8")

print(
    "허용오차 옵션 적용:",
    '"--atol"' in pipeline_text
    and '"3e-4"' in pipeline_text,
)

허용오차 옵션 적용: True


In [34]:
from datetime import datetime, timezone

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_root = REPOSITORY_ROOT / "outputs" / f"colab_{CATEGORY}_{run_id}"
pipeline_command = [
    sys.executable, "src/run_colab_pipeline.py",
    "--checkpoint", str(checkpoint),
    "--training-manifest", str(training_manifest),
    "--dataset-root", str(REPOSITORY_ROOT / "datasets" / "MVTecAD"),
    "--category", CATEGORY,
    "--output-root", str(output_root),
    "--calibration-images", str(CALIBRATION_IMAGES),
    "--device", "cuda",
]
print("실행 명령:", " ".join(pipeline_command))
pipeline_log = output_root.with_suffix(".log")
pipeline_log.parent.mkdir(parents=True, exist_ok=True)
with pipeline_log.open("w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        pipeline_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
    return_code = process.wait()
if return_code != 0:
    print("파이프라인 실패 로그:", pipeline_log)
    raise RuntimeError(
        f"run_colab_pipeline.py 실패(return code={return_code}). "
        f"위의 첫 ERROR/Traceback을 확인하세요. 전체 로그: {pipeline_log}"
    )
print("파이프라인 완료:", output_root)

실행 명령: /usr/bin/python3 src/run_colab_pipeline.py --checkpoint /content/edge-anomaly-detection/models/patchcore_bottle.ckpt --training-manifest /content/edge-anomaly-detection/results/metadata/train_bottle.json --dataset-root /content/edge-anomaly-detection/datasets/MVTecAD --category bottle --output-root /content/edge-anomaly-detection/outputs/colab_bottle_20261002T130946209766Z --calibration-images 100 --device cuda
+ /usr/bin/python3 /content/edge-anomaly-detection/src/evaluate.py --checkpoint /content/edge-anomaly-detection/models/patchcore_bottle.ckpt --dataset-root /content/edge-anomaly-detection/datasets/MVTecAD --category bottle --output-csv /content/edge-anomaly-detection/outputs/colab_bottle_20261002T130946209766Z/results/csv/accuracy.csv --manifest /content/edge-anomaly-detection/outputs/colab_bottle_20261002T130946209766Z/results/metadata/evaluate_bottle.json --accelerator gpu
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importin

In [35]:

  export_text = (
      REPOSITORY_ROOT / "src" / "export_onnx.py"
  ).read_text(encoding="utf-8")

  pipeline_text = (
      REPOSITORY_ROOT / "src" / "run_colab_pipeline.py"
  ).read_text(encoding="utf-8")

  print(
      "TorchScript exporter 적용:",
      '"dynamo": False' in export_text,
  )

  print(
      "CPU export 적용:",
      '"cpu"' in pipeline_text,
  )

  print(
      "허용오차 옵션 적용:",
      '"--atol"' in pipeline_text
      and '"3e-4"' in pipeline_text,
  )


TorchScript exporter 적용: True
CPU export 적용: True
허용오차 옵션 적용: True


In [36]:
  export_text = (
      REPOSITORY_ROOT / "src" / "export_onnx.py"
  ).read_text(encoding="utf-8")

  print("TorchScript exporter 적용:", '"dynamo": False' in export_text)

TorchScript exporter 적용: True


In [37]:

from pathlib import Path

log_path = Path(
    "/content/edge-anomaly-detection/outputs/"
    "colab_bottle_20261002T125456082592Z.log"
)

lines = log_path.read_text(
    encoding="utf-8",
    errors="replace",
).splitlines()

start = next(
    index
    for index, line in enumerate(lines)
    if line.startswith("Traceback (most recent call last)")
)

print("\n".join(lines[start:start + 40]))


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/exporter/_core.py", line 1460, in export
    decomposed_program = _prepare_exported_program_for_export(
        program, registry=registry
    )
  File "/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/exporter/_core.py", line 1011, in _prepare_exported_program_for_export
    _fx_passes.insert_type_promotion_nodes(graph_module)
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/exporter/_fx_passes.py", line 29, in insert_type_promotion_nodes
    passes.InsertTypePromotion(module).run()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/fx/_pass.py", line 237, in run
    return self._run(*args, **kwargs)
           ~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/fx/passes/type_promotion.py"

In [38]:
from pathlib import Path

log_path = Path(
    "/content/edge-anomaly-detection/outputs/"
    "colab_bottle_20261002T120404326109Z.log"
)

lines = log_path.read_text(
    encoding="utf-8",
    errors="replace",
).splitlines()

traceback_indices = [
    index
    for index, line in enumerate(lines)
    if line.startswith("Traceback (most recent call last)")
]

print("발견한 traceback 수:", len(traceback_indices))

for number, start in enumerate(traceback_indices, start=1):
    print(f"\n===== TRACEBACK {number} =====")
    end = min(start + 100, len(lines))
    print("\n".join(lines[start:end]))

발견한 traceback 수: 2

===== TRACEBACK 1 =====
Traceback (most recent call last):
  File "/content/edge-anomaly-detection/src/export_onnx.py", line 332, in <module>
    main()
    ~~~~^^
  File "/content/edge-anomaly-detection/src/export_onnx.py", line 263, in main
    raise RuntimeError(
    ...<2 lines>...
    )
RuntimeError: PyTorch/ONNX mismatch: max_abs_error=0.000245094299, mean_abs_error=1.9150782e-06
Traceback (most recent call last):
  File "/content/edge-anomaly-detection/src/run_colab_pipeline.py", line 216, in <module>
    main()
    ~~~~^^
  File "/content/edge-anomaly-detection/src/run_colab_pipeline.py", line 118, in main
    run(
    ~~~^
        [
        ^
    ...<16 lines>...
        ]
        ^
    )
    ^
  File "/content/edge-anomaly-detection/src/run_colab_pipeline.py", line 43, in run
    subprocess.run(command, check=True)
    ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/subprocess.py", line 577, in run
    raise CalledProcessError(retcode, proc

## 9. bundle 검증

In [39]:
bundle = output_root / f"{CATEGORY}_artifacts.zip"
checksum = bundle.with_suffix(bundle.suffix + ".sha256")
subprocess.run(
    [sys.executable, "src/verify_artifact_bundle.py", "--bundle", str(bundle)],
    check=True,
)
if not checksum.is_file():
    raise FileNotFoundError(f"Bundle checksum이 없습니다: {checksum}")
print("검증 완료:", bundle)
print("Checksum:", checksum)

검증 완료: /content/edge-anomaly-detection/outputs/colab_bottle_20261002T130946209766Z/bottle_artifacts.zip
Checksum: /content/edge-anomaly-detection/outputs/colab_bottle_20261002T130946209766Z/bottle_artifacts.zip.sha256


## 10. 검증된 결과를 Google Drive에 저장

In [40]:
drive_run_output = DRIVE_OUTPUT_ROOT / run_id
drive_run_output.mkdir(parents=True, exist_ok=False)
drive_bundle = shutil.copy2(bundle, drive_run_output / bundle.name)
drive_checksum = shutil.copy2(checksum, drive_run_output / checksum.name)
print("Jetson 전달 파일:")
print(drive_bundle)
print(drive_checksum)

Jetson 전달 파일:
/content/drive/MyDrive/edge-anomaly-output/20261002T130946209766Z/bottle_artifacts.zip
/content/drive/MyDrive/edge-anomaly-output/20261002T130946209766Z/bottle_artifacts.zip.sha256


## 11. 선택: 브라우저로 직접 다운로드

Drive 저장만 필요하면 이 셀은 실행하지 않아도 됩니다.

In [41]:
# 필요할 때 아래 주석을 해제하세요.
# from google.colab import files
# files.download(str(bundle))
# files.download(str(checksum))